# CAPM & Fama-French 3 Facteurs — CAC 40
### Beta, Alpha de Jensen, SMB et HML sur les 40 valeurs de l'indice français

Ce notebook applique les deux grands modèles de pricing des actifs financiers
aux **40 composantes exactes du CAC 40** avec des données réelles via **yfinance**
et les facteurs européens de la **bibliothèque Ken French**.

| Modèle | Formule | Facteurs |
|--------|---------|----------|
| **CAPM** | $r_i - r_f = \alpha + \beta(r_m - r_f) + \varepsilon$ | Marché |
| **Fama-French** | $r_i - r_f = \alpha + \beta^{Mkt}(r_m-r_f) + \beta^{SMB}\cdot SMB + \beta^{HML}\cdot HML + \varepsilon$ | Marché + Taille + Valeur |

**Plan :** données → performance → corrélations (heatmaps) → CAPM → Fama-French → synthèse

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yfinance as yf
import statsmodels.api as sm
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titlesize": 13})

## 1. Les 40 composantes du CAC 40

In [ ]:
# ── Composition exacte du CAC 40 (40 valeurs) ──────────────────────────────
CAC40 = {
    # Luxe / Mode
    "MC.PA":    "Luxe",           # LVMH
    "RMS.PA":   "Luxe",           # Hermès
    "KER.PA":   "Luxe",           # Kering
    # Consommation / Distribution
    "OR.PA":    "Consommation",   # L'Oréal
    "BN.PA":    "Consommation",   # Danone
    "RI.PA":    "Consommation",   # Pernod Ricard
    "CA.PA":    "Consommation",   # Carrefour
    # Finance & Assurance
    "BNP.PA":   "Finance",        # BNP Paribas
    "ACA.PA":   "Finance",        # Crédit Agricole
    "GLE.PA":   "Finance",        # Société Générale
    "CS.PA":    "Finance",        # AXA
    # Énergie
    "TTE.PA":   "Énergie",        # TotalEnergies
    "ENGI.PA":  "Énergie",        # Engie
    # Santé
    "SAN.PA":   "Santé",          # Sanofi
    "EL.PA":    "Santé",          # EssilorLuxottica
    "ERF.PA":   "Santé",          # Eurofins Scientific
    # Industrie / Aérospatiale / Défense
    "AIR.PA":   "Industrie",      # Airbus
    "SAF.PA":   "Industrie",      # Safran
    "HO.PA":    "Industrie",      # Thales
    "SU.PA":    "Industrie",      # Schneider Electric
    "LR.PA":    "Industrie",      # Legrand
    "DG.PA":    "Industrie",      # Vinci
    "EN.PA":    "Industrie",      # Bouygues
    "ALO.PA":   "Industrie",      # Alstom
    "AI.PA":    "Industrie",      # Air Liquide
    # Technologie
    "CAP.PA":   "Technologie",    # Capgemini
    "DSY.PA":   "Technologie",    # Dassault Systèmes
    "STM.PA":   "Technologie",    # STMicroelectronics
    # Automobile
    "RNO.PA":   "Automobile",     # Renault
    "STLAM.PA": "Automobile",     # Stellantis
    "ML.PA":    "Automobile",     # Michelin
    # Matériaux
    "MT.PA":    "Matériaux",      # ArcelorMittal
    "SGO.PA":   "Matériaux",      # Saint-Gobain
    # Services aux collectivités
    "VIE.PA":   "Services",       # Veolia
    "EDEN.PA":  "Services",       # Edenred
    # Télécom
    "ORA.PA":   "Télécom",        # Orange
    # Communication / Médias
    "PUB.PA":   "Communication",  # Publicis
    "VIV.PA":   "Communication",  # Vivendi
    # Hôtellerie
    "AC.PA":    "Hôtellerie",     # Accor
    # Immobilier
    "URW.PA":   "Immobilier",     # Unibail-Rodamco-Westfield
}

SECTOR_COLORS = {
    "Luxe":          "#c0392b",
    "Finance":       "#2980b9",
    "Industrie":     "#27ae60",
    "Technologie":   "#8e44ad",
    "Énergie":       "#e67e22",
    "Santé":         "#16a085",
    "Consommation":  "#f39c12",
    "Matériaux":     "#7f8c8d",
    "Automobile":    "#d35400",
    "Services":      "#1abc9c",
    "Télécom":       "#2c3e50",
    "Communication": "#e74c3c",
    "Hôtellerie":    "#9b59b6",
    "Immobilier":    "#34495e",
}

tickers = list(CAC40.keys())
sectors = pd.Series(CAC40, name="Secteur")
print(f"Nombre de valeurs CAC 40 : {len(CAC40)}")
print("\nRépartition par secteur :")
print(sectors.value_counts().to_string())

In [ ]:
START = "2019-01-01"
END   = "2024-12-31"

raw = yf.download(tickers + ["^FCHI"], start=START, end=END,
                  auto_adjust=True, progress=False)["Close"]
raw = raw.rename(columns={"^FCHI": "CAC40"})
raw = raw.ffill().dropna(how="all")

# Garder uniquement les colonnes avec moins de 10% de NaN
raw = raw.loc[:, raw.isnull().mean() < 0.10]
available = [t for t in tickers if t in raw.columns]
sectors   = sectors[sectors.index.isin(available)]

prices = raw[available + ["CAC40"]].dropna()
returns    = prices.pct_change().dropna()
stock_ret  = returns[available]
market_ret = returns["CAC40"]

print(f"Période    : {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Valeurs    : {len(available)} / 40 téléchargées avec succès")
print(f"Jours      : {len(prices)} jours de cotation")
if len(available) < 40:
    missing = [t for t in tickers if t not in available]
    print(f"Non dispo  : {missing}")

## 2. Performance normalisée (base 100 = jan. 2019)

In [ ]:
base = prices / prices.iloc[0] * 100

from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(15, 6))
for ticker in available:
    sect  = CAC40.get(ticker, "Autre")
    color = SECTOR_COLORS.get(sect, "gray")
    ax.plot(base.index, base[ticker], linewidth=0.9, alpha=0.5, color=color)

ax.plot(base.index, base["CAC40"], color="black", linewidth=2.5,
        label="CAC 40", zorder=5)
ax.axhline(100, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Performance normalisée — 40 valeurs CAC 40 (base 100 = jan. 2019)")
ax.set_ylabel("Indice")

legend_elems = [Line2D([0],[0], color=c, linewidth=2, label=s)
                for s, c in SECTOR_COLORS.items() if s in sectors.values]
legend_elems.append(Line2D([0],[0], color="black", linewidth=2.5, label="CAC 40 (indice)"))
ax.legend(handles=legend_elems, ncol=3, fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## 3. Statistiques descriptives

In [ ]:
RF_ANNUAL = 0.03
rf_daily  = RF_ANNUAL / 252

ann_ret = stock_ret.mean() * 252
ann_vol = stock_ret.std()  * np.sqrt(252)
sharpe  = ann_ret / ann_vol

stats = pd.DataFrame({
    "Rendement (%)":    (ann_ret * 100).round(2),
    "Volatilité (%)":   (ann_vol * 100).round(2),
    "Sharpe":           sharpe.round(2),
    "Secteur":          sectors,
}).sort_values("Sharpe", ascending=False)

print("Top 10 par Sharpe Ratio")
print(stats.head(10).to_string())
print("\nFlop 5")
print(stats.tail(5).to_string())

## 4. Matrice de corrélation — Tous les titres (triée par secteur)

In [ ]:
corr = stock_ret.corr()

# Trier les titres par secteur
order = sectors.sort_values().index.tolist()
order = [t for t in order if t in corr.index]
corr_sorted = corr.loc[order, order]
short_labels = [t.replace(".PA", "") for t in order]

# Masque triangle supérieur
mask = np.triu(np.ones_like(corr_sorted, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    corr_sorted, mask=mask,
    cmap="RdYlGn", vmin=-0.2, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 6.5},
    linewidths=0.3, linecolor="white",
    xticklabels=short_labels, yticklabels=short_labels,
    ax=ax, cbar_kws={"shrink": 0.55, "label": "Corrélation"}
)
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=7.5)
ax.set_yticklabels(short_labels, rotation=0, fontsize=7.5)
ax.set_title("Matrice de corrélation — Rendements journaliers CAC 40\n"
             "(triangle inférieur, trié par secteur)", pad=14)

# Séparateurs entre secteurs
sect_vals = sectors[order].values
for i in range(1, len(sect_vals)):
    if sect_vals[i] != sect_vals[i-1]:
        ax.axhline(i, color="navy", linewidth=1.8)
        ax.axvline(i, color="navy", linewidth=1.8)

plt.tight_layout()
plt.show()

## 5. Corrélation inter-sectorielle

In [ ]:
# Rendement moyen journalier par secteur
sect_daily = stock_ret.copy()
sect_daily.columns = sectors[sect_daily.columns]
sector_avg  = sect_daily.T.groupby(level=0).mean().T
sector_corr = sector_avg.corr()

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# ── A : heatmap corrélation sectorielle ─────────────────────────────────────
sns.heatmap(
    sector_corr, ax=axes[0],
    cmap="RdYlGn", vmin=-0.2, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 9},
    linewidths=0.8, square=True,
    cbar_kws={"shrink": 0.7}
)
axes[0].set_title("Corrélation inter-sectorielle\n(rendements journaliers moyens)")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right", fontsize=9)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, fontsize=9)

# ── B : performance cumulée par secteur ─────────────────────────────────────
cumul_sector = (1 + sector_avg).cumprod()
for col in cumul_sector.columns:
    axes[1].plot(cumul_sector.index, cumul_sector[col],
                 linewidth=2, label=col, color=SECTOR_COLORS.get(col, "gray"))
axes[1].axhline(1, color="black", linestyle="--", linewidth=0.8)
axes[1].set_title("Performance cumulée par secteur (base 1 = jan. 2019)")
axes[1].set_ylabel("Croissance de 1 €")
axes[1].legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.show()

## 6. Corrélation glissante 90 jours avec le CAC 40

In [ ]:
roll_corr = pd.DataFrame({
    s: sector_avg[s].rolling(90).corr(market_ret)
    for s in sector_avg.columns
}).dropna()

fig, ax = plt.subplots(figsize=(14, 5))
for s in roll_corr.columns:
    ax.plot(roll_corr.index, roll_corr[s], linewidth=1.5,
            label=s, color=SECTOR_COLORS.get(s, "gray"))
ax.axhline(0, color="black", linewidth=0.6)
ax.set_ylim(-0.3, 1.1)
ax.set_title("Corrélation glissante 90 jours — Secteur vs CAC 40")
ax.set_ylabel("Corrélation")
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## 7. CAPM — Beta et Alpha de Jensen

$$r_i - r_f = \alpha_i + \beta_i\,(r_m - r_f) + \varepsilon_i$$

- **β > 1** : titre cyclique, amplifie les mouvements du marché
- **β < 1** : titre défensif, amortit les chocs
- **α > 0** : surperformance résiduelle non rémunérée par le risque systématique

In [ ]:
excess_market = market_ret - rf_daily
capm_results  = {}

for ticker in available:
    exc = stock_ret[ticker] - rf_daily
    model = sm.OLS(exc, sm.add_constant(excess_market)).fit()
    capm_results[ticker] = {
        "alpha_ann": model.params["const"] * 252,
        "beta":      model.params["CAC40"],
        "r2":        model.rsquared,
        "p_alpha":   model.pvalues["const"],
    }

capm_df = pd.DataFrame(capm_results).T
capm_df["secteur"]      = sectors
capm_df["ticker_short"] = capm_df.index.str.replace(".PA", "", regex=False)

print("CAPM results — triés par beta décroissant")
print(capm_df[["secteur","beta","alpha_ann","r2"]].round(4)
      .sort_values("beta", ascending=False).to_string())

## 8. Security Market Line (SML)

In [ ]:
market_premium = float(market_ret.mean() * 252 - RF_ANNUAL)
beta_range     = np.linspace(capm_df["beta"].min() - 0.15,
                              capm_df["beta"].max() + 0.15, 200)

fig, ax = plt.subplots(figsize=(13, 8))
ax.plot(beta_range, (RF_ANNUAL + beta_range * market_premium) * 100,
        color="black", linewidth=1.8, linestyle="--",
        label="SML théorique", zorder=2)

for sect in sectors.unique():
    mask = capm_df["secteur"] == sect
    sub  = capm_df[mask]
    ax.scatter(sub["beta"], ann_ret[sub.index] * 100,
               color=SECTOR_COLORS.get(sect, "gray"),
               s=100, edgecolors="white", linewidths=0.5,
               zorder=3, label=sect)
    for _, row in sub.iterrows():
        ax.annotate(row["ticker_short"],
                    (row["beta"], ann_ret[row.name] * 100),
                    fontsize=7, ha="left", va="bottom",
                    xytext=(3, 3), textcoords="offset points")

ax.axhline(RF_ANNUAL * 100, color="gray", linestyle=":", linewidth=1,
           label=f"Taux sans risque ({RF_ANNUAL:.0%})")
ax.axvline(1, color="gray",  linestyle=":", linewidth=1)
ax.set_title("Security Market Line — CAC 40  (2019–2024)\n"
             "Au-dessus de la SML = alpha positif")
ax.set_xlabel("Beta β")
ax.set_ylabel("Rendement annualisé réel (%)")
ax.legend(ncol=2, fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## 9. Heatmaps CAPM — Beta et Alpha par secteur

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

# ── A : beta moyen par secteur ──────────────────────────────────────────────
beta_by_sect  = capm_df.groupby("secteur")["beta"].mean().sort_values(ascending=False)
axes[0].barh(beta_by_sect.index, beta_by_sect.values,
             color=[SECTOR_COLORS.get(s,"gray") for s in beta_by_sect.index],
             edgecolor="white")
axes[0].axvline(1, color="black", linestyle="--", linewidth=1.2)
axes[0].set_title("Beta moyen par secteur")
axes[0].set_xlabel("β moyen")

# ── B : alpha moyen par secteur ──────────────────────────────────────────────
alpha_by_sect = capm_df.groupby("secteur")["alpha_ann"].mean().sort_values(ascending=False)
axes[1].barh(alpha_by_sect.index, alpha_by_sect.values * 100,
             color=["green" if v > 0 else "crimson" for v in alpha_by_sect.values],
             edgecolor="white")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Alpha de Jensen annualisé moyen (%)")
axes[1].set_xlabel("α (%)")

# ── C : heatmap beta titre × secteur ────────────────────────────────────────
pivot = capm_df.pivot_table(index="secteur", columns="ticker_short",
                             values="beta", aggfunc="first")
pivot = pivot.reindex(sorted(pivot.index))
sns.heatmap(pivot, ax=axes[2], cmap="RdYlGn_r", center=1,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=0.4, cbar_kws={"shrink": 0.7},
            mask=pivot.isnull())
axes[2].set_title("Beta par titre et secteur")
axes[2].set_xlabel("")
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha="right", fontsize=7)
axes[2].set_yticklabels(axes[2].get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
plt.show()

## 10. Classement Alpha — Créateurs et destructeurs de valeur

In [ ]:
alpha_sorted = capm_df.sort_values("alpha_ann")

fig, ax = plt.subplots(figsize=(13, 9))
bar_colors = [SECTOR_COLORS.get(alpha_sorted.loc[i,"secteur"],"gray")
              for i in alpha_sorted.index]
bars = ax.barh(alpha_sorted["ticker_short"], alpha_sorted["alpha_ann"] * 100,
               color=bar_colors, edgecolor="white", height=0.7)
ax.axvline(0, color="black", linewidth=1)

for bar, val in zip(bars, alpha_sorted["alpha_ann"] * 100):
    ax.text(val + (0.15 if val >= 0 else -0.15),
            bar.get_y() + bar.get_height() / 2,
            f"{val:+.1f}%", va="center",
            ha="left" if val >= 0 else "right", fontsize=7.5)

ax.set_title("Alpha de Jensen annualisé — 40 valeurs CAC 40\n"
             "(CAPM, rf = 3%, marché = indice CAC 40, 2019–2024)")
ax.set_xlabel("α annualisé (%)")
plt.tight_layout()
plt.show()

## 11. Fama-French 3 Facteurs — Europe

$$r_i - r_f = \alpha_i + \beta_i^{Mkt}(r_m-r_f) + \beta_i^{SMB}\cdot SMB + \beta_i^{HML}\cdot HML + \varepsilon_i$$

| Facteur | Signification | Implication pratique |
|---------|--------------|----------------------|
| **Mkt-RF** | Prime de risque marché | Risque systématique |
| **SMB** | *Small Minus Big* — petites caps vs grandes caps | Exposition à l'effet taille |
| **HML** | *High Minus Low* — valeur vs croissance | Exposition à l'effet valeur (book-to-market) |

In [ ]:
try:
    import pandas_datareader.data as web
    ff_raw = web.DataReader("Europe_3_Factors_Daily", "famafrench",
                             start=START, end=END)
    ff = ff_raw[0].copy() / 100
    ff.index = pd.to_datetime(ff.index)
    ff.columns = ["Mkt-RF", "SMB", "HML", "RF"]
    print(f"✅ Facteurs FF Europe Daily téléchargés : {len(ff)} obs.")
    FF_SOURCE = "Fama-French Europe Daily — Ken French Data Library"

except Exception as e:
    print(f"⚠️  Téléchargement FF échoué ({e})\n   → Construction de proxies depuis les données CAC 40")
    vol_rank    = stock_ret.std().rank()
    small_t     = vol_rank.nlargest(10).index
    large_t     = vol_rank.nsmallest(10).index
    momentum    = (prices[available].iloc[-1] / prices[available].iloc[0] - 1)
    value_t     = momentum.nsmallest(10).index
    growth_t    = momentum.nlargest(10).index
    ff = pd.DataFrame({
        "Mkt-RF": market_ret - rf_daily,
        "SMB":    stock_ret[small_t].mean(axis=1) - stock_ret[large_t].mean(axis=1),
        "HML":    stock_ret[value_t].mean(axis=1) - stock_ret[growth_t].mean(axis=1),
        "RF":     rf_daily,
    }, index=market_ret.index)
    FF_SOURCE = "Proxies construits depuis les données CAC 40"

print(f"Source : {FF_SOURCE}")

In [ ]:
common = stock_ret.index.intersection(ff.index)
sr, fa = stock_ret.loc[common], ff.loc[common]

ff_results = {}
for ticker in available:
    y = sr[ticker] - fa["RF"]
    model = sm.OLS(y, sm.add_constant(fa[["Mkt-RF","SMB","HML"]])).fit()
    ff_results[ticker] = {
        "alpha_ann":  model.params["const"] * 252,
        "beta_mkt":   model.params["Mkt-RF"],
        "beta_smb":   model.params["SMB"],
        "beta_hml":   model.params["HML"],
        "r2_adj":     model.rsquared_adj,
    }

ff_df = pd.DataFrame(ff_results).T
ff_df["secteur"]      = sectors
ff_df["ticker_short"] = ff_df.index.str.replace(".PA","",regex=False)

print("Fama-French — Top 10 alpha annualisé")
print(ff_df[["secteur","alpha_ann","beta_mkt","beta_smb","beta_hml","r2_adj"]]
      .round(4).sort_values("alpha_ann", ascending=False).head(10).to_string())

## 12. Heatmap des chargements Fama-French — Tous les titres

In [ ]:
# Trier par secteur
ff_sorted = ff_df.sort_values(["secteur","ticker_short"])
lm = ff_sorted.set_index("ticker_short")[["beta_mkt","beta_smb","beta_hml"]]
lm.columns = ["β Mkt-RF", "β SMB", "β HML"]

fig, ax = plt.subplots(figsize=(9, max(8, len(lm) * 0.45)))
sns.heatmap(lm, ax=ax, cmap="RdYlGn", center=0,
            annot=True, fmt=".2f", annot_kws={"size": 8},
            linewidths=0.5, cbar_kws={"shrink": 0.55})
ax.set_title("Chargements Fama-French par titre\n(trié par secteur)")
ax.set_xticklabels(["β Marché", "β Taille (SMB)", "β Valeur (HML)"], rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

# Séparateurs secteur
sv = ff_sorted["secteur"].values
for i in range(1, len(sv)):
    if sv[i] != sv[i-1]:
        ax.axhline(i, color="navy", linewidth=1.8)

plt.tight_layout()
plt.show()

## 13. Heatmap des chargements moyens par secteur

In [ ]:
load_sect = ff_df.groupby("secteur")[["beta_mkt","beta_smb","beta_hml","alpha_ann"]].mean()
load_sect.columns = ["β Mkt-RF", "β SMB", "β HML", "α ann."]
load_sect["α ann."] *= 100

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# ── A : chargements β ────────────────────────────────────────────────────────
sns.heatmap(load_sect.drop(columns="α ann."), ax=axes[0],
            cmap="RdYlGn", center=0, annot=True, fmt=".3f",
            annot_kws={"size": 10}, linewidths=0.8,
            cbar_kws={"shrink": 0.7})
axes[0].set_title("Chargements Fama-French moyens par secteur")
axes[0].set_xticklabels(["β Marché", "β Taille (SMB)", "β Valeur (HML)"], rotation=0)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)

# ── B : alpha CAPM vs FF ─────────────────────────────────────────────────────
a_comp = pd.DataFrame({
    "CAPM (%)":         capm_df.groupby("secteur")["alpha_ann"].mean() * 100,
    "Fama-French (%)":  load_sect["α ann."],
}).sort_values("CAPM (%)")

x = np.arange(len(a_comp))
axes[1].barh(x - 0.2, a_comp["CAPM (%)"],         0.38, label="Alpha CAPM",
             color="steelblue", edgecolor="white")
axes[1].barh(x + 0.2, a_comp["Fama-French (%)"],  0.38, label="Alpha Fama-French",
             color="coral",     edgecolor="white")
axes[1].set_yticks(x)
axes[1].set_yticklabels(a_comp.index, fontsize=9)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Alpha CAPM vs Fama-French par secteur (%)")
axes[1].set_xlabel("Alpha annualisé (%)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 14. Tableau de bord final — 6 panneaux

In [ ]:
r2_comp = pd.DataFrame({
    "R² CAPM":         capm_df.groupby("secteur")["r2"].mean(),
    "R² Fama-French":  ff_df.groupby("secteur")["r2_adj"].mean(),
})

fig = plt.figure(figsize=(19, 13))
gs  = gridspec.GridSpec(2, 3, hspace=0.42, wspace=0.32)

# ── 1 : beta par titre ───────────────────────────────────────────────────────
ax1  = fig.add_subplot(gs[0, 0])
cd   = capm_df.sort_values("beta")
ax1.barh(cd["ticker_short"], cd["beta"],
         color=[SECTOR_COLORS.get(r["secteur"],"gray") for _,r in cd.iterrows()],
         edgecolor="white", height=0.7)
ax1.axvline(1, color="black", linestyle="--", linewidth=1)
ax1.set_title("1. Beta CAPM par titre", fontweight="bold")
ax1.tick_params(labelsize=6.5)

# ── 2 : alpha FF par titre ───────────────────────────────────────────────────
ax2  = fig.add_subplot(gs[0, 1])
fd   = ff_df.sort_values("alpha_ann")
ax2.barh(fd["ticker_short"], fd["alpha_ann"] * 100,
         color=["green" if v > 0 else "crimson" for v in fd["alpha_ann"]],
         edgecolor="white", height=0.7)
ax2.axvline(0, color="black", linewidth=0.8)
ax2.set_title("2. Alpha Fama-French annualisé (%)", fontweight="bold")
ax2.tick_params(labelsize=6.5)

# ── 3 : R² CAPM vs FF par secteur ────────────────────────────────────────────
ax3  = fig.add_subplot(gs[0, 2])
x3   = np.arange(len(r2_comp))
ax3.bar(x3-0.2, r2_comp["R² CAPM"]*100,        0.38, label="CAPM",         color="steelblue", edgecolor="white")
ax3.bar(x3+0.2, r2_comp["R² Fama-French"]*100, 0.38, label="Fama-French",  color="coral",     edgecolor="white")
ax3.set_xticks(x3)
ax3.set_xticklabels(r2_comp.index, rotation=45, ha="right", fontsize=8)
ax3.set_title("3. R² moyen : CAPM vs FF (%)", fontweight="bold")
ax3.legend(fontsize=8)

# ── 4 : corrélation sectorielle ──────────────────────────────────────────────
ax4  = fig.add_subplot(gs[1, 0])
sns.heatmap(sector_corr, ax=ax4, cmap="RdYlGn", vmin=-0.2, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=0.5, square=True, cbar=False)
ax4.set_title("4. Corrélation inter-sectorielle", fontweight="bold")
ax4.tick_params(labelsize=7)
ax4.set_xticklabels(ax4.get_xticklabels(), rotation=45, ha="right")

# ── 5 : chargements FF sectoriels ────────────────────────────────────────────
ax5  = fig.add_subplot(gs[1, 1])
sns.heatmap(load_sect.drop(columns="α ann."), ax=ax5,
            cmap="RdYlGn", center=0, annot=True, fmt=".2f",
            annot_kws={"size": 8}, linewidths=0.5, cbar=False)
ax5.set_title("5. Chargements FF par secteur", fontweight="bold")
ax5.set_xticklabels(["β Mkt","β SMB","β HML"], rotation=0, fontsize=9)
ax5.tick_params(labelsize=8)

# ── 6 : performance cumulée sectorielle ──────────────────────────────────────
ax6  = fig.add_subplot(gs[1, 2])
for col in cumul_sector.columns:
    ax6.plot(cumul_sector.index, cumul_sector[col],
             linewidth=1.8, label=col, color=SECTOR_COLORS.get(col,"gray"))
ax6.axhline(1, color="black", linestyle="--", linewidth=0.8)
ax6.set_title("6. Performance cumulée par secteur", fontweight="bold")
ax6.legend(ncol=2, fontsize=7)

fig.suptitle("CAC 40 — Tableau de bord CAPM & Fama-French (2019–2024)",
             fontsize=16, fontweight="bold", y=1.01)
plt.show()